# Datadog Remote MCP Server as an AgentCore Gateway Target (API key)

This is the **API-key variant** of the OAuth 2.1 3LO notebook (`01-...`). It connects the
**Datadog Remote MCP Server** to **Amazon Bedrock AgentCore Gateway** as an MCP target, but
authenticates to Datadog with a **Datadog API key + application key** instead of per-user OAuth.

Use this simpler path when a **single shared identity** is acceptable (internal automation, demos).
Unlike the 3LO notebook there is **no Dynamic Client Registration and no browser Authorize step** —
but you also lose per-user RBAC/audit: every call runs as the identity that owns the keys.

### The two-header constraint (why this isn't a one-line change)

Datadog's MCP server requires **two** headers — `DD_API_KEY` and `DD_APPLICATION_KEY`. An AgentCore
Gateway target allows only **one** credential provider, which injects **one** header. So we:
1. inject `DD_API_KEY` via an **API-key credential provider**, and
2. forward `DD_APPLICATION_KEY` from the client through **header propagation**
   (`metadataConfiguration.allowedRequestHeaders`).

| Information | Details |
|-------------|---------|
| Tutorial type | Interactive |
| AgentCore components | AgentCore Gateway, AgentCore Identity |
| Agentic framework | Strands Agents |
| Gateway target type | MCP server |
| Inbound auth IdP | Amazon Cognito (JWT) |
| Outbound auth | **API key + application key** (provider + header propagation) |
| LLM model | Anthropic Claude Sonnet 4.6 |
| SDK | boto3 |
| Vertical | Observability |
| Complexity | Intermediate |

## Prerequisites

- An AWS account with access to Amazon Bedrock AgentCore (Gateway + Identity).
- Permissions to create Cognito user pools, AgentCore gateways, targets, and credential providers.
- A **Datadog API key** and **application key**. The application key's user/role needs the
  **`mcp_read`** permission (and `mcp_write` for write tools) — the Datadog Standard Role has these
  by default. Scope the keys to a service account with least privilege.
  See [API and Application Keys](https://docs.datadoghq.com/account_management/api-app-keys/).
- Model access to **Anthropic Claude Sonnet 4.6** in Amazon Bedrock.
- Python 3.10+.

Install dependencies:

In [ ]:
!pip install -r requirements.txt -q

## Datadog Remote MCP Server — key facts (API-key auth)

(Sources: [Datadog MCP setup](https://docs.datadoghq.com/bits_ai/mcp_server/setup/),
[official repo](https://github.com/datadog-labs/mcp-server).)

- **MCP endpoint (US1):** `https://mcp.datadoghq.com/api/unstable/mcp-server/mcp`
  (swap host for other sites: `mcp.datadoghq.eu`, etc.).
- **Auth:** send `DD_API_KEY` and `DD_APPLICATION_KEY` as HTTP headers. Available to **any** Datadog
  customer — no employee/internal restriction, no OAuth flow.
- **Toolset scoping:** append `?toolsets=core,llmobs` to the endpoint. Stick to GA toolsets;
  `apm`, `code-exec`, and `rum` are Preview and need a signup form.
- **Not supported** on `ddog-gov` sites or `us2`.

In [ ]:
import boto3, json, os, requests

# ---- Configuration -------------------------------------------------------
REGION = "us-east-2"  # your AgentCore region

# Datadog credentials — read from env (recommended) or paste here.
DD_API_KEY = os.environ.get("DD_API_KEY", "<YOUR_DD_API_KEY>")
DD_APPLICATION_KEY = os.environ.get("DD_APPLICATION_KEY", "<YOUR_DD_APPLICATION_KEY>")

DD_MCP_ENDPOINT = "https://mcp.datadoghq.com/api/unstable/mcp-server/mcp?toolsets=core,llmobs"

# Distinct names from the 3LO notebook so both can coexist in one account.
GATEWAY_NAME = "datadog-mcp-apikey-gateway"
TARGET_NAME  = "datadog-mcp"
POOL_NAME    = "datadog-mcp-apikey-pool"
MCP_PROTOCOL_VERSION = "2025-11-25"

os.environ.setdefault("AWS_DEFAULT_REGION", REGION)  # utils.py reads the session region

agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
identity  = boto3.client("bedrock-agentcore-control", region_name=REGION)

## Step 0 — Create the IAM role the Gateway will assume

Same shared helper as the 3LO notebook. Requires `iam:CreateRole` / `iam:PutRolePolicy`.

In [ ]:
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import utils

gateway_role = utils.create_agentcore_gateway_role("datadog-mcp-apikey-gw")
GATEWAY_ROLE_ARN = gateway_role["Role"]["Arn"]
print("Gateway role ARN:", GATEWAY_ROLE_ARN)

## Step 1 — Create the API-key credential provider (for DD_API_KEY)

This stores the Datadog **API key** in AgentCore Identity. The target (Step 3) references it and
injects it as the `DD_API_KEY` header on every outbound call. The **application key** is handled
separately via header propagation (Step 3) — it can't go through the same provider.

> Method/field names for the API-key provider can vary by `bedrock-agentcore` SDK version. If this
> call's signature differs, create the API-key credential provider via the AgentCore console and
> set `apikey_provider_arn` to its ARN.

In [ ]:
apikey_provider = identity.create_api_key_credential_provider(
    name="datadog-mcp-apikey",
    apiKey=DD_API_KEY,
)
apikey_provider_arn = apikey_provider["credentialProviderArn"]
print("API-key provider ARN:", apikey_provider_arn)

## Step 2 — Create the Gateway (Cognito JWT inbound)

Inbound auth is Cognito JWT, same as the 3LO notebook (kept for parity). Note: without 3LO you
*could* also use IAM (SigV4) inbound and skip Cognito entirely — that restriction only applies to 3LO.

In [ ]:
cognito = boto3.client("cognito-idp", region_name=REGION)
pool = cognito.create_user_pool(PoolName=POOL_NAME)
USER_POOL_ID = pool["UserPool"]["Id"]
app_client = cognito.create_user_pool_client(
    UserPoolId=USER_POOL_ID, ClientName="datadog-mcp-apikey-client",
    GenerateSecret=False, ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
)
APP_CLIENT_ID = app_client["UserPoolClient"]["ClientId"]
DISCOVERY = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
print("User pool:", USER_POOL_ID, "| app client:", APP_CLIENT_ID)

In [ ]:
gw = agentcore.create_gateway(
    name=GATEWAY_NAME,
    roleArn=GATEWAY_ROLE_ARN,
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {
            "supportedVersions": [MCP_PROTOCOL_VERSION],
            "searchType": "SEMANTIC",
        }
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {"discoveryUrl": DISCOVERY, "allowedClients": [APP_CLIENT_ID]}
    },
)
GATEWAY_ID = gw["gatewayId"]
GATEWAY_URL = gw["gatewayUrl"]
print("Gateway:", GATEWAY_ID, GATEWAY_URL)

## Step 3 — Create the MCP target (API-key provider + header propagation)

Two pieces of outbound auth wiring:
1. **`credentialProviderConfigurations`** → the API-key provider injects the `DD_API_KEY` header.
2. **`metadataConfiguration.allowedRequestHeaders`** → allows `DD_APPLICATION_KEY` to be forwarded
   from the inbound client request through to Datadog (header propagation).

Because there's no 3LO, the target should reach **READY** without any Authorize step.

> Header-propagation field names (`metadataConfiguration` / `allowedRequestHeaders`) and the API-key
> provider sub-shape may vary by SDK version — print the response and adjust if a key is rejected.

In [ ]:
target = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=TARGET_NAME,
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": DD_MCP_ENDPOINT}}},
    credentialProviderConfigurations=[{
        "credentialProviderType": "API_KEY",
        "credentialProvider": {
            "apiKeyCredentialProvider": {
                "providerArn": apikey_provider_arn,
                "credentialLocation": "HEADER",
                "credentialParameterName": "DD_API_KEY",
            }
        },
    }],
    # Forward the application key from the client request to Datadog.
    metadataConfiguration={"allowedRequestHeaders": ["DD_APPLICATION_KEY"]},
)
TARGET_ID = target["targetId"]
print("Target:", TARGET_ID, "| status:", target.get("status"))

In [ ]:
# Wait for the target to be READY (no Authorize step needed for API-key auth).
import time
while True:
    s = agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)["items"][0]["status"]
    print(s)
    if s in ("READY", "FAILED"):
        break
    time.sleep(5)

## Step 4 — Invoke the Gateway (list Datadog tools)

Each call sends: the Cognito **JWT** (inbound), the `MCP-Protocol-Version` header, and the
**`DD_APPLICATION_KEY`** header (which the gateway forwards to Datadog). The gateway adds
`DD_API_KEY` itself from the credential provider.

In [ ]:
# Create + confirm a Cognito test user for the inbound JWT.
import secrets
TEST_USER = "datadog-mcp-apikey-user"
TEST_PASSWORD = "Test@" + secrets.token_urlsafe(12) + "1!"
cognito.admin_create_user(UserPoolId=USER_POOL_ID, Username=TEST_USER,
                          MessageAction="SUPPRESS", TemporaryPassword=TEST_PASSWORD)
cognito.admin_set_user_password(UserPoolId=USER_POOL_ID, Username=TEST_USER,
                                Password=TEST_PASSWORD, Permanent=True)
print("Created + confirmed Cognito user:", TEST_USER)

In [ ]:
auth = cognito.initiate_auth(
    ClientId=APP_CLIENT_ID, AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": TEST_USER, "PASSWORD": TEST_PASSWORD},
)
JWT = auth["AuthenticationResult"]["AccessToken"]

HEADERS = {
    "Authorization": f"Bearer {JWT}",
    "Content-Type": "application/json",
    "MCP-Protocol-Version": MCP_PROTOCOL_VERSION,
    "DD_APPLICATION_KEY": DD_APPLICATION_KEY,  # forwarded to Datadog via header propagation
}
resp = requests.post(
    GATEWAY_URL, headers=HEADERS,
    json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}, timeout=60,
)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2)[:4000])

## Step 5 — Use the tools from a Strands agent

The agent connects over streamable HTTP, sending the same headers (including `DD_APPLICATION_KEY`).

> Note: with `searchType=SEMANTIC`, `list_tools_sync()` returns only the
> `x_amz_bedrock_agentcore_search` proxy, so the agent can *discover* Datadog tools but Strands may
> not let it *invoke* them. To have the agent call tools directly, recreate the gateway without
> `searchType` so `tools/list` returns the Datadog tools.

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

def dd_mcp_client():
    return streamablehttp_client(GATEWAY_URL, headers={
        "Authorization": f"Bearer {JWT}",
        "MCP-Protocol-Version": MCP_PROTOCOL_VERSION,
        "DD_APPLICATION_KEY": DD_APPLICATION_KEY,
    })

mcp_client = MCPClient(dd_mcp_client)
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Discovered {len(tools)} tools through the gateway")
    agent = Agent(model="us.anthropic.claude-sonnet-4-6", tools=tools)
    result = agent("What are the open P1 incidents and any recent error-rate spikes in my Datadog org?")
    print(result)

## Cleanup

In [ ]:
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# agentcore.delete_gateway(gatewayIdentifier=GATEWAY_ID)
# identity.delete_api_key_credential_provider(name="datadog-mcp-apikey")
# cognito.delete_user_pool(UserPoolId=USER_POOL_ID)

## API key vs OAuth 3LO

This notebook trades per-user governance for simplicity:

| | API key (this notebook) | OAuth 3LO (`01-...`) |
|---|---|---|
| Setup | No DCR, no browser Authorize | DCR + per-user browser Authorize |
| Identity | Single shared (key owner) | Per-user |
| Datadog RBAC / audit | Coarse (one identity) | Per-user RBAC, DAC, audit |
| Best for | Demos, internal automation | Multi-user agent platforms |

Both respect Datadog RBAC; the difference is whether it's enforced per-user or per shared identity.